# N135 · 树状数组与前缀聚合

**目标：** 利用lowbit分解前缀并实现动态求和。

**先修：** N124, N134, N031。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 一基下标；lowbit；点增量；前缀/区间和；双BIT扩展；逆序计数。

## 从问题到算法

Fenwick把前缀切成若干长度为lowbit(i)的块。内部采用一基下标，bit[i]保存原数组区间[i−lowbit(i),i)的和。查询不断清除最低1，把前缀分成不重叠块；更新不断加lowbit，走过所有覆盖该位置的块。公共接口仍使用零基下标与半开边界。

## 最小实现

**本章接口：** `Fenwick.add(i, delta)`、`Fenwick.prefix_sum(end)`、`count_smaller(nums)`

In [1]:
class Fenwick:
    def __init__(self,values):
        values=[0]*values if isinstance(values,int) else list(values)
        self.n=len(values); self.bit=[0]+values
        for i in range(1,self.n+1):
            parent=i+(i&-i)
            if parent<=self.n: self.bit[parent]+=self.bit[i]
    def add(self,i,delta):
        if not 0<=i<self.n: raise IndexError('point index outside tree')
        i+=1
        while i<=self.n: self.bit[i]+=delta; i+=i&-i
    def prefix_sum(self,end):
        if not 0<=end<=self.n: raise IndexError('prefix boundary outside tree')
        answer=0
        while end: answer+=self.bit[end]; end-=end&-end
        return answer
    def range_sum(self,left,right):
        if left>right: raise ValueError('invalid range')
        return self.prefix_sum(right)-self.prefix_sum(left)

def count_smaller(nums):
    ordered=sorted(set(nums)); ranks={x:i for i,x in enumerate(ordered)}; tree=Fenwick(len(ordered)); answer=[]
    for x in reversed(nums):
        r=ranks[x]; answer.append(tree.prefix_sum(r)); tree.add(r,1)
    return answer[::-1]

## 正确性、前提与复杂度

bit[i]的覆盖块是二进制下标分解决定的。查询沿i−lowbit(i)得到互不重叠且恰覆盖目标前缀的块；点增量更新所有包含该点的祖先块，维持块和不变量。较小数查询用prefix_sum(rank)，不包含等值秩。

**代价：** 线性建树O(n)；更新和前缀/区间查询O(log n)；计数含压缩O(n log n)，空间O(n)。add是增量而非赋值；修改旧值3到新值2应传−1。

## 小实例与可视化

先手算，再运行。下表由当前内核中的代码和显式小实例生成；不是预制答案图片。

In [2]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

a=[1,3,5,7,9,11]; tree=Fenwick(a)
show_table(['内部下标','覆盖原数组半开区间','块和'],[(i,(i-(i&-i),i),tree.bit[i]) for i in range(1,len(a)+1)])

内部下标,覆盖原数组半开区间,块和
1,"(0, 1)",1
2,"(0, 2)",4
3,"(2, 3)",5
4,"(0, 4)",16
5,"(4, 5)",9
6,"(4, 6)",20


## 自动测试

以下断言检查实现契约。测试通过不等同于一般性证明，也不表示已经提交 LeetCode 官方评测。

In [3]:
tree=Fenwick([1,3,5]); assert tree.range_sum(0,3)==9
tree.add(1,-1); assert tree.range_sum(0,3)==8
assert count_smaller([5,2,6,1])==[2,1,1,0]
from random import Random
rng=Random(135)
for n in range(1,25):
    a=[rng.randrange(-5,6) for _ in range(n)]; tree=Fenwick(a)
    for _ in range(80):
        if rng.randrange(2):
            i=rng.randrange(n); delta=rng.randrange(-5,6); a[i]+=delta; tree.add(i,delta)
        else:
            l=rng.randrange(n+1); r=rng.randrange(l,n+1)
            assert tree.range_sum(l,r)==sum(a[l:r])
    assert count_smaller(a)==[sum(y<x for y in a[i+1:]) for i,x in enumerate(a)]
assert Fenwick(0).prefix_sum(0)==0
print('N135: 所有本章断言通过')

N135: 所有本章断言通过


## 练习：先独立完成

**练习 1：** 实现范围加与点查。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

**练习 2：** 更新赋值时先计算增量。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

未完成练习不阻断教学区 Run All。验收时解释为什么正确，不只展示运行截图。

## 代表题与迁移

- 307. Range Sum Query - Mutable（canonical）
- 315. Count of Smaller Numbers After Self（canonical）
- 493. Reverse Pairs（extension）

这些题用于知识映射；本章实现遵循上文明确的教学契约。除原规格注明已核对身份的条目外，本轮未重新联网核验全部题面、收费权限或平台语言支持。不要把同概念教学实例当作所有官方题的完整答案。

## 自测与下一步

不看代码，复述状态、不变量和一个失效条件；改变一个输入约束，判断实现是否仍成立。确认能解释边界测试后，按 `specs/curriculum.json` 的 `learning_order` 进入下一章。